# Phase 5 — Rigour: bootstrap CIs + multi-seed


In [ ]:
# --- Setup: mount Drive + install driftbench from the public repo ---
from google.colab import drive  # noqa
drive.mount('/content/drive')

# Install the pinned commit for reproducibility (replace USERNAME and the ref).
!pip -q install "git+https://github.com/USERNAME/drift-conference.git@main"

import os
os.environ['DRIFT_DATA_ROOT'] = '/content/drive/MyDrive/drift-conference/data'
os.environ['DRIFT_REPO_ROOT'] = '/content/drift-conference'  # for results/manifests output
import driftbench as db; print('driftbench', db.__version__)


Bootstrap 95% CIs on the random-vs-chronological macro-F1 **gap**, and multi-seed mean±std for the model-dependent results. Report gaps as `0.36 [0.31, 0.41]`, never bare point estimates.

In [ ]:
import pandas as pd, numpy as np
from sklearn.metrics import f1_score
from driftbench import (config, make_model, random_split, chronological_split,
                        bootstrap_gap_ci, multi_seed, fmt_ci)
ds='CSE-CIC-IDS2018'; MODEL='rf'
X=pd.read_parquet(config.INTERIM_DIR/ds/'features.parquet')
y=pd.read_parquet(config.INTERIM_DIR/ds/'labels.parquet')['label']
meta=pd.read_parquet(config.INTERIM_DIR/ds/'metadata.parquet')
macro = lambda yt,yp: f1_score(yt,yp,average='macro',zero_division=0)


In [ ]:
def run(seed):
    rs=random_split(len(X), y, seed=seed)
    cs=chronological_split(meta)
    mr=make_model(MODEL,seed).fit(X.iloc[rs['train']], y.iloc[rs['train']])
    mc=make_model(MODEL,seed).fit(X.iloc[cs['train']], y.iloc[cs['train']])
    pr=mr.predict(X.iloc[rs['test']]); pc=mc.predict(X.iloc[cs['test']])
    return {'random_macroF1':macro(y.iloc[rs['test']],pr),
            'chrono_macroF1':macro(y.iloc[cs['test']],pc)}

summary = multi_seed(run, config.SEEDS)
for k,v in summary.items(): print(f"{k}: {v['mean']:.3f} ± {v['std']:.3f}")


In [ ]:
# Bootstrap CI on the gap (anchor seed).
rs=random_split(len(X), y, seed=config.SEED_ANCHOR); cs=chronological_split(meta)
mr=make_model(MODEL,config.SEED_ANCHOR).fit(X.iloc[rs['train']], y.iloc[rs['train']])
mc=make_model(MODEL,config.SEED_ANCHOR).fit(X.iloc[cs['train']], y.iloc[cs['train']])
gap=bootstrap_gap_ci(
    {'y_true':y.iloc[rs['test']].values,'y_pred':mr.predict(X.iloc[rs['test']])},
    {'y_true':y.iloc[cs['test']].values,'y_pred':mc.predict(X.iloc[cs['test']])},
    macro)
print('random - chrono macro-F1 gap:', fmt_ci(gap['gap'], gap['lo'], gap['hi']))
